# Manuscript Figures — Big5Loop (MDPI Style)

This notebook generates all figures referenced in **manuscript_v2.md** for the thesis paper *Personality-Aware Digital Coaching Through Adaptive Dialogue*.

**MDPI compliance:**
- Resolution: ≥300 DPI (and/or ≥1000 px width/height)
- Clear when viewed at up to 11 × 9 cm
- Numbering and captions in manuscript; no captions inside figure files

**Output:** All figures are saved under `Big5Loop/visualization/figures/`.

## 1. Setup — MDPI style and paths

In [3]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# --- Paths (run from Big5Loop/visualization or project root) ---
NOTEBOOK_DIR = Path(".").resolve()
if NOTEBOOK_DIR.name != "visualization":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "Big5Loop" / "visualization"
FIG_DIR = NOTEBOOK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_ROOT = NOTEBOOK_DIR.parent / "evaluation_data"

# --- MDPI figure style (300 DPI, clear at small size) ---
MDPI_DPI = 300
MDPI_FIG_WIDTH_SINGLE = 3.54   # ~89 mm single column
MDPI_FIG_WIDTH_DOUBLE = 7.08   # ~178 mm double column
plt.rcParams.update({
    "figure.dpi": MDPI_DPI,
    "savefig.dpi": MDPI_DPI,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

OCEAN_ORDER = ["O", "C", "E", "A", "N"]
OCEAN_LABELS = {
    "O": "Openness", "C": "Conscientiousness", "E": "Extraversion",
    "A": "Agreeableness", "N": "Neuroticism"
}

print(f"Figures will be saved to: {FIG_DIR}")

Figures will be saved to: /Users/huaduojiejia/MyProject/hslu/2026/thesis project/Big5Loop/visualization/figures


## 2. Figure 1 — System Architecture

Three-tier architecture: API route → N8N 9-stage workflow → PostgreSQL/pgvector; pre-processing and post-processing flows.

In [ ]:
from IPython.display import Image, display
import subprocess

DOT_PATH = NOTEBOOK_DIR / "figure1_architecture.dot"
PNG_PATH = FIG_DIR / "figure1_system_architecture.png"
SVG_PATH = FIG_DIR / "figure1_system_architecture.svg"

subprocess.run(["dot", "-Tpng", str(DOT_PATH), "-o", str(PNG_PATH)], check=True)
subprocess.run(["dot", "-Tsvg", str(DOT_PATH), "-o", str(SVG_PATH)], check=True)

display(Image(filename=str(PNG_PATH)))
print(f"Saved: {PNG_PATH}")
print(f"Saved: {SVG_PATH}")


## 3. Figure 2 — EMA Trait Convergence

Neuroticism: raw per-turn detections vs EMA-smoothed trajectory; confidence gating at one turn (e.g. turn 3).

In [ ]:
def ema_smooth(observations, confidences, alpha_s=0.3, tau_c=0.4, prior=0.0):
    """In-session EMA with confidence gating."""
    out = [prior]
    for obs, c in zip(observations, confidences):
        if c >= tau_c:
            out.append(alpha_s * obs + (1 - alpha_s) * out[-1])
        else:
            out.append(out[-1])  # reject low-confidence
    return np.array(out[1:])

def draw_figure2_ema_convergence(out_path: Path):
    np.random.seed(42)
    turns = np.arange(1, 11)
    true_level = 0.5  # underlying high Neuroticism
    raw = true_level + np.random.randn(10) * 0.35
    raw = np.clip(raw, -1, 1)
    conf = np.clip(0.6 + np.random.randn(10) * 0.2, 0.2, 1.0)
    conf[2] = 0.25  # low confidence at turn 3 → gating
    ema_vals = ema_smooth(raw, conf, alpha_s=0.3, tau_c=0.4, prior=0.0)

    fig, ax = plt.subplots(figsize=(MDPI_FIG_WIDTH_SINGLE, 2.8))
    ax.plot(turns, raw, "o-", color="#E74C3C", markersize=6, label="Raw detection", alpha=0.8)
    ax.plot(turns, ema_vals, "s-", color="#2980B9", markersize=5, label="EMA-smoothed")
    ax.axhline(true_level, color="gray", linestyle="--", alpha=0.7, label="Underlying trait")
    ax.axvline(3, color="orange", linestyle=":", alpha=0.8, label="Low conf. (rejected)")
    ax.set_xlabel("Turn")
    ax.set_ylabel("Neuroticism estimate")
    ax.set_ylim(-0.2, 1.1)
    ax.legend(loc="lower right", fontsize=7)
    ax.set_title("EMA Trait Convergence (Neuroticism)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=MDPI_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {out_path}")

draw_figure2_ema_convergence(FIG_DIR / "figure2_ema_convergence.png")

## 4. Figure 3 — Trait stabilization trajectories

EMA-smoothed trajectories for representative OCEAN traits across 6+ turns (synthetic profiles).

In [ ]:
def draw_figure3_trait_trajectories(out_path: Path):
    np.random.seed(123)
    turns = np.arange(1, 9)
    # Two synthetic profiles: high-N and balanced
    profiles = {
        "High Neuroticism": {"N": 0.6, "others": 0.0},
        "Balanced": {"N": 0.0, "others": 0.0},
    }
    fig, axes = plt.subplots(1, 2, figsize=(MDPI_FIG_WIDTH_DOUBLE * 0.6, 2.6))
    colors = plt.cm.tab10(np.linspace(0, 1, 5))
    for ax, (profile_name, levels) in zip(axes, profiles.items()):
        for i, trait in enumerate(OCEAN_ORDER):
            base = levels["N"] if trait == "N" else levels["others"]
            obs = base + np.cumsum(np.random.randn(8) * 0.08)
            obs = np.clip(obs, -1, 1)
            ema = ema_smooth(obs, np.ones(8) * 0.7, alpha_s=0.3, prior=base)
            ax.plot(turns, ema, "-", color=colors[i], label=OCEAN_LABELS[trait], linewidth=1.5)
        ax.set_xlabel("Turn")
        ax.set_ylabel("Trait estimate")
        ax.set_title(profile_name)
        ax.legend(loc="right", fontsize=6)
        ax.set_ylim(-0.5, 0.9)
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out_path, dpi=MDPI_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {out_path}")

draw_figure3_trait_trajectories(FIG_DIR / "figure3_trait_trajectories.png")

## 5. Figure 4 — Coaching quality comparison

Adaptive (personality-aware) vs three baselines across three dimensions: emotional tone, relevance/coherence, emotional needs responsiveness.

In [ ]:
def draw_figure4_coaching_quality(out_path: Path):
    dimensions = ["Emotional tone\nappropriateness", "Relevance and\ncoherence", "Emotional needs\nresponsiveness"]
    conditions = ["Personality-aware", "Generic", "Memory-only", "Policy-only"]
    # Placeholder means (replace with real evaluation when available)
    data = np.array([
        [4.2, 4.1, 4.3],   # Personality-aware
        [3.2, 3.4, 3.0],   # Generic
        [3.6, 3.7, 3.5],   # Memory-only
        [3.3, 3.8, 3.2],   # Policy-only
    ])
    x = np.arange(len(dimensions))
    width = 0.2
    fig, ax = plt.subplots(figsize=(MDPI_FIG_WIDTH_DOUBLE * 0.85, 3.0))
    colors = ["#2980B9", "#BDC3C7", "#F39C12", "#95A5A6"]
    for i, (cond, row) in enumerate(zip(conditions, data)):
        ax.bar(x + (i - 1.5) * width, row, width, label=cond, color=colors[i], edgecolor="#333", linewidth=0.5)
    ax.set_ylabel("Score (1–5)")
    ax.set_xticks(x)
    ax.set_xticklabels(dimensions)
    ax.legend(loc="upper right", ncol=2, fontsize=7)
    ax.set_ylim(0, 5.2)
    ax.axhline(4.0, color="gray", linestyle=":", alpha=0.6)
    ax.set_title("Coaching Quality by Condition")
    plt.tight_layout()
    plt.savefig(out_path, dpi=MDPI_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {out_path}")

draw_figure4_coaching_quality(FIG_DIR / "figure4_coaching_quality.png")

## 6. Evaluation-data figures (manuscript style)

When evaluation data exists, generate MDPI-style versions of:
- PERSONAGE: detected vs ground truth OCEAN (scatter per trait)
- BIG5-CHAT: agreement by trait (bar chart)

In [ ]:
def load_personage_results():
    path = EVAL_ROOT / "personage" / "processed" / "personage_eval_results.jsonl"
    if not path.exists():
        path = EVAL_ROOT / "personage" / "processed" / "personage_eval.jsonl"
    if not path.exists():
        return None
    rows = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            if r.get("detected_ocean") and r.get("ground_truth_ocean"):
                rows.append(r)
    return rows if rows else None

def load_big5_agreement_data():
    path = EVAL_ROOT / "processed" / "eval_results.jsonl"
    if not path.exists():
        return None
    rows = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            if r.get("detected_ocean") and r.get("trait") and r.get("level"):
                rows.append(r)
    return rows if rows else None

In [ ]:
personage_rows = load_personage_results()
big5_rows = load_big5_agreement_data()
print(f"PERSONAGE samples: {len(personage_rows) if personage_rows else 0}")
print(f"BIG5-CHAT samples: {len(big5_rows) if big5_rows else 0}")

In [ ]:
if personage_rows:
    # Detected vs ground truth scatter (one subplot per OCEAN trait), MDPI style
    data = {k: {"det": [], "gt": []} for k in OCEAN_ORDER}
    for r in personage_rows:
        det = r.get("detected_ocean") or {}
        gt = r.get("ground_truth_ocean") or {}
        for k in OCEAN_ORDER:
            data[k]["det"].append(float(det.get(k, 0)))
            data[k]["gt"].append(float(gt.get(k, 0)))
    fig, axes = plt.subplots(2, 3, figsize=(MDPI_FIG_WIDTH_DOUBLE, 4))
    axes = axes.flatten()
    for idx, k in enumerate(OCEAN_ORDER):
        ax = axes[idx]
        det = np.array(data[k]["det"])
        gt = np.array(data[k]["gt"])
        ax.scatter(gt, det, alpha=0.6, s=25, c="#2980B9", edgecolors="white", linewidth=0.3)
        ax.plot([-1, 1], [-1, 1], "r--", alpha=0.5)
        r_val = np.corrcoef(gt, det)[0, 1] if len(gt) >= 2 and np.std(gt) > 1e-6 and np.std(det) > 1e-6 else np.nan
        ax.set_title(f"{OCEAN_LABELS[k]}" + (f", r = {r_val:.3f}" if not np.isnan(r_val) else ""))
        ax.set_xlabel("Ground truth")
        ax.set_ylabel("Detected")
        ax.set_xlim(-1.1, 1.1)
        ax.set_ylim(-1.1, 1.1)
    axes[-1].axis("off")
    plt.suptitle("PERSONAGE: Detected vs Ground Truth OCEAN")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eval_personage_detected_vs_gt.png", dpi=MDPI_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved: figures/eval_personage_detected_vs_gt.png")
else:
    print("No PERSONAGE results found; skip eval figure.")

In [ ]:
TRAIT_TO_OCEAN = {"openness": "O", "conscientiousness": "C", "extraversion": "E", "agreeableness": "A", "neuroticism": "N"}

if big5_rows:
    trait_stats = {t: {"agree": 0, "disagree": 0} for t in TRAIT_TO_OCEAN}
    for r in big5_rows:
        trait = r.get("trait")
        level = r.get("level")
        ocean = r.get("detected_ocean") or {}
        if not trait or trait not in TRAIT_TO_OCEAN:
            continue
        key = TRAIT_TO_OCEAN[trait]
        det = float(ocean.get(key, 0))
        expected = 1 if level == "high" else -1
        det_sign = 1 if det > 0 else (-1 if det < 0 else 0)
        if det_sign == expected:
            trait_stats[trait]["agree"] += 1
        elif det_sign != 0:
            trait_stats[trait]["disagree"] += 1
    traits = list(TRAIT_TO_OCEAN.keys())
    agrees = [trait_stats[t]["agree"] for t in traits]
    disagrees = [trait_stats[t]["disagree"] for t in traits]
    totals = [a + d for a, d in zip(agrees, disagrees)]
    rates = [a / t * 100 if t > 0 else 0 for a, t in zip(agrees, totals)]
    fig, ax = plt.subplots(figsize=(MDPI_FIG_WIDTH_SINGLE, 2.8))
    x = np.arange(len(traits))
    w = 0.35
    ax.bar(x - w/2, agrees, w, label="Agree", color="#2980B9")
    ax.bar(x + w/2, disagrees, w, label="Disagree", color="#E74C3C", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([OCEAN_LABELS[TRAIT_TO_OCEAN[t]] for t in traits], fontsize=8)
    ax.set_ylabel("Count")
    ax.set_title("BIG5-CHAT: Agreement by Trait")
    ax.legend(fontsize=7)
    for i, (r, t) in enumerate(zip(rates, totals)):
        if t > 0:
            ax.annotate(f"{r:.0f}%", (i, agrees[i] + 0.3), ha="center", fontsize=7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eval_agreement_by_trait.png", dpi=MDPI_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved: figures/eval_agreement_by_trait.png")
else:
    print("No BIG5-CHAT results found; skip agreement figure.")

## Summary

| Figure | File | Description |
|--------|------|-------------|
| 1 | `figure1_system_architecture.png` | Three-tier Big5Loop architecture |
| 2 | `figure2_ema_convergence.png` | EMA convergence (Neuroticism) |
| 3 | `figure3_trait_trajectories.png` | Trait stabilization across turns |
| 4 | `figure4_coaching_quality.png` | Coaching quality by condition |
| Eval | `eval_personage_detected_vs_gt.png` | PERSONAGE detected vs GT (if data present) |
| Eval | `eval_agreement_by_trait.png` | BIG5-CHAT agreement by trait (if data present) |